# Module 7 — Bedrock at the Edges: NL↔SPARQL with Guardrails

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

The ontology is loaded, the data is promoted, the SHACL shapes enforce the boundary.
But a wealth advisor or a compliance officer does not write SPARQL queries. They ask
questions in plain English: "Which customers generated a large-deposit signal in the
last 90 days?"

This module builds the **natural-language-to-SPARQL** component that translates
human questions into graph queries. The LLM (Large Language Model) does one thing:
translate. It does not reason, score, route, or make decisions.

By the end of this module you can:

- Build a few-shot NL-to-SPARQL component grounded in the FIBO-aligned ontology
- Validate every LLM-generated query against SHACL shapes before execution
- Explain why the LLM is confined to translation (not reasoning) in ATLAS
- Ask the seven competency questions in plain English and get correct answers

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **NL↔SPARQL** | Natural-Language-to-SPARQL translation. The process of converting a human question into a SPARQL query that a graph database can execute. The LLM does the translation; the graph does the answering. |
| **Few-shot prompting** | Giving the LLM a small number of examples ("shots") of correct question-to-SPARQL pairs before asking it to translate a new question. More examples = better accuracy. |
| **Grounding** | Constraining the LLM's output to use only vocabulary that exists in the ontology. A grounded query uses `atlas:Customer` (which exists) rather than `atlas:Client` (which does not). |
| **SHACL pre-check** | Running SHACL validation on a query's implied writes BEFORE executing it. If the query would insert probabilistic-opaque data, the pre-check catches it. |
| **Amazon Bedrock** | AWS's managed service for foundation models. Provides API access to LLMs (like Anthropic Claude) without managing infrastructure. |
| **Guardrails** | Bedrock Guardrails — configurable filters that redact PII (Personally Identifiable Information) from LLM outputs and block harmful content. |
| **Prompt scaffold** | The structured prompt template that tells the LLM how to behave: what ontology vocabulary to use, what format to output, what to avoid. |
| **Ground-truth pairs** | Known-correct question/SPARQL pairs used as few-shot examples. The LLM sees these before translating a new question. |
| **tips.yaml** | FIBO-specific hints for the LLM: which prefixes to use, which classes exist, common query patterns. |
| **prefixes.txt** | The standard SPARQL prefix block that every generated query must include. |

## The LLM does three things and only three things

This is the architectural boundary for Bedrock in ATLAS:

1. **Translates** natural-language questions into SPARQL queries grounded in the
   ontology and validated against SHACL shapes before execution
2. **Explains** reasoner outputs in plain English to a human reader, with the
   reasoner's inference chain as the source of truth
3. **Drafts** narratives (case summaries, referral cover notes) for human review
   before any external action is taken

The LLM does NOT:
- Infer relationships
- Score risk
- Pick routes by reasoning over a case
- Approve workflows
- Make compliance decisions

## Prerequisites

- Module 6 complete (SHACL shapes in place)
- Amazon Bedrock enabled in us-east-1 with access to Anthropic Claude models
- The SageMaker execution role has `bedrock:InvokeModel` permission

## Deliverables

- `prompts/tips.yaml` — FIBO-specific hints for the LLM
- `prompts/ground-truth.yaml` — 15–20 few-shot competency-question/SPARQL pairs
- `prompts/prefixes.txt` — standard SPARQL prefix block
- A working NL-to-SPARQL component with SHACL pre-check
- All seven competency questions answerable in plain English

## Architecture class for this module

**PROBABILISTIC-OPAQUE at the translation layer.** The LLM's translation is
non-deterministic (the same question may produce slightly different SPARQL across
runs). However, the output is validated deterministically (SHACL pre-check +
syntax parse) before execution. The LLM is confined to an interface role — it
translates between human language and structured queries. It does not reason.

## How This Connects to Competency Questions

This is where the Competency Questions (CQs) from Module 1 take on their second
and third roles. In Module 1, CQs served as **validation** — acceptance tests that
proved the ontology had the right structure. Here in Module 7, the same seven
Competency Questions serve two additional purposes:

**Grounding.** The ground-truth pairs that teach the LLM (Large Language Model)
how to generate SPARQL are the same seven CQs from Module 1, each paired with its
known-correct SPARQL query. The Competency Questions constrain what vocabulary the
LLM is allowed to use. If a concept isn't needed to answer any CQ, it shouldn't
appear in generated queries.

**Accuracy.** When you measure whether the NL-to-SPARQL component works correctly,
you measure it against the CQ ground-truth pairs. Did the generated query return
the same results as the known-correct query? Competency Questions give you a
measurable definition of "correct" that isn't subjective.

The lifecycle of a single Competency Question across this workshop:

| Module | CQ Role | What Happens |
|--------|---------|-------------|
| 1 | **Validation** | Written as an acceptance test — proves ontology structure |
| 2 | **Stability** | Remains valid after FIBO alignment |
| 4 | **Data requirement** | Defines what data must be loaded |
| 5 | **Derivation** | Answered by computed data for the first time |
| 6 | **Enforcement** | SHACL enforces what CQs imply |
| 7 | **Grounding + Accuracy** | Becomes few-shot examples and benchmarks for the LLM |
| 8 | **Proof of value** | Answered end-to-end in the CIO demo |

In [ ]:
import sys
sys.path.insert(0, '../notebooks/shared')

import json
import boto3
from pathlib import Path
import atlas_sparql

print('Module 7 — Bedrock at the Edges')
print(f'SPARQL validator loaded: atlas_sparql.validate()')

# Bedrock client
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
BEDROCK_MODEL = 'us.anthropic.claude-sonnet-4-6'
print(f'Bedrock model: {BEDROCK_MODEL}')

# Load the prefix block
PREFIXES = atlas_sparql.build_prefixes()
print(f'\nSPARQL prefixes loaded ({len(PREFIXES.splitlines())} lines)')

## The Ground-Truth Pairs

Few-shot prompting works by showing the LLM examples of correct translations before
asking it to translate a new question. The more examples, the better the accuracy.

The ground-truth pairs below map each of the seven competency questions to a correct
SPARQL query. These are the "answer key" the LLM learns from.

### Why ground-truth pairs matter

Without examples, the LLM will:
- Invent property names that do not exist in the ontology (`atlas:hasWealth` instead of `atlas:producesSignal`)
- Use wrong prefixes (`fibo:Customer` instead of `atlas:Customer`)
- Generate syntactically valid but semantically wrong queries

With examples, the LLM learns the vocabulary, the patterns, and the style of
correct ATLAS SPARQL queries. It is pattern-matching, not reasoning — and that
is exactly what we want for a translation task.

In [ ]:
# Ground-truth pairs: competency question -> correct SPARQL
GROUND_TRUTH = [
    {
        'question': 'Which customers have generated a wealth signal in the last 90 days?',
        'sparql': PREFIXES + '''SELECT ?customer ?signalType ?signalDate WHERE {
    ?customer a atlas:Customer ;
              atlas:producesSignal ?sig .
    ?sig atlas:hasSignalType ?signalType ;
         atlas:signalDate ?signalDate ;
         atlas:withinWindow ?window .
}'''
    },
    {
        'question': 'For a given signal, what observations support it and what is the score?',
        'sparql': PREFIXES + '''SELECT ?signal ?txn ?scoreValue ?isProbabilistic WHERE {
    ?signal a atlas:WealthSignal ;
            atlas:evidencedBy ?txn ;
            atlas:hasScore ?score .
    ?score atlas:scoreValue ?scoreValue ;
           atlas:probabilistic ?isProbabilistic .
}'''
    },
    {
        'question': 'Which household relationships does this customer have?',
        'sparql': PREFIXES + '''SELECT ?customer ?household WHERE {
    ?customer a atlas:Customer ;
              atlas:memberOf ?household .
}'''
    },
    {
        'question': 'Has this customer been previously surfaced as a wealth candidate?',
        'sparql': PREFIXES + '''SELECT ?customer ?prevSurfacing WHERE {
    ?customer a atlas:Customer ;
              atlas:hasPreviousSurfacing ?prevSurfacing .
}'''
    },
    {
        'question': 'Which routing decisions have human review and what was the outcome?',
        'sparql': PREFIXES + '''SELECT ?routing ?route ?review ?outcome WHERE {
    ?routing a atlas:RoutingDecision ;
             atlas:selectedRoute ?route ;
             atlas:reviewedBy ?review .
    ?review atlas:reviewOutcome ?outcome .
}'''
    },
    {
        'question': 'What is the full audit trail from customer to advisor approval?',
        'sparql': PREFIXES + '''SELECT ?customer ?eligibility ?routing ?review ?advisor WHERE {
    ?customer atlas:hasEligibility ?eligibility .
    ?eligibility atlas:triggersRouting ?routing .
    ?routing atlas:reviewedBy ?review .
    ?review atlas:conductedBy ?advisor .
}'''
    },
    {
        'question': 'What specific transactions were used to surface this customer?',
        'sparql': PREFIXES + '''SELECT ?customer ?signal ?txn ?txnDate ?amount WHERE {
    ?customer a atlas:Customer ;
              atlas:producesSignal ?signal .
    ?signal atlas:evidencedBy ?txn .
    ?txn atlas:transactionDate ?txnDate ;
         atlas:amountUSD ?amount .
}'''
    },
]

print(f'Ground-truth pairs loaded: {len(GROUND_TRUTH)}')
print()
for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f'  CQ{i}: {gt["question"][:60]}...')

## The NL-to-SPARQL Translation Component

The cell below defines the core translation function. It:
1. Builds a prompt with the ground-truth pairs as few-shot examples
2. Sends the user's question to Bedrock
3. Extracts the SPARQL from the response
4. Validates the SPARQL syntax via `atlas_sparql.validate()`
5. Returns the validated query (or raises an error if invalid)

### The SHACL pre-check

If the generated query contains INSERT or UPDATE statements (which it should not
for read-only questions), the pre-check runs SHACL validation on the implied writes.
This catches any attempt by the LLM to generate a query that would write
probabilistic-opaque data to the SLGD.

In [ ]:
def nl_to_sparql(question: str, verbose: bool = False) -> str:
    """Translate a natural-language question to a SPARQL query using Bedrock.
    
    The LLM is grounded in the ATLAS ontology via few-shot examples.
    Every generated query is validated before being returned.
    
    Parameters
    ----------
    question : str
        The natural-language question to translate.
    verbose : bool
        If True, print the full prompt and response.
    
    Returns
    -------
    str
        A validated SPARQL query string.
    
    Raises
    ------
    ValueError
        If the LLM generates invalid SPARQL or the query fails validation.
    """
    # Build the few-shot prompt
    examples = ''
    for gt in GROUND_TRUTH[:5]:  # Use first 5 as examples
        examples += f'Question: {gt["question"]}\nSPARQL:\n{gt["sparql"]}\n\n'
    
    prompt = f"""You are a SPARQL query generator for the ATLAS financial services ontology.
Your ONLY job is to translate natural-language questions into SPARQL SELECT queries.

Rules:
- Use ONLY the prefixes and properties defined in the examples below
- Generate ONLY SELECT queries (never INSERT, UPDATE, or DELETE)
- Use the exact property names from the ontology (atlas:producesSignal, not atlas:hasSignal)
- Always include the PREFIX declarations
- Return ONLY the SPARQL query, no explanation

Ontology vocabulary available:
Classes: atlas:Customer, atlas:Account, atlas:Transaction, atlas:WealthSignal, atlas:Score,
         atlas:Eligibility, atlas:RoutingDecision, atlas:HumanReview, atlas:Advisor,
         atlas:Household, atlas:HouseholdMembership, atlas:ObservationWindow
Properties: atlas:producesSignal, atlas:hasScore, atlas:evidencedBy, atlas:hasSignalType,
            atlas:memberOf, atlas:hasEligibility, atlas:triggersRouting, atlas:reviewedBy,
            atlas:conductedBy, atlas:hasAccount, atlas:hasTransaction, atlas:scoreValue,
            atlas:probabilistic, atlas:confidence, atlas:transactionDate, atlas:amountUSD,
            atlas:selectedRoute, atlas:reviewOutcome, atlas:promotedFrom

Examples:
{examples}

Now translate this question:
Question: {question}
SPARQL:
"""
    
    if verbose:
        print(f'Prompt length: {len(prompt)} chars')
    
    # Call Bedrock
    response = bedrock.invoke_model(
        modelId=BEDROCK_MODEL,
        contentType='application/json',
        accept='application/json',
        body=json.dumps({
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': 500,
            'messages': [{'role': 'user', 'content': prompt}]
        })
    )
    result = json.loads(response['body'].read())
    sparql_output = result['content'][0]['text'].strip()
    
    # Clean up: remove markdown code fences if present
    if sparql_output.startswith('```'):
        sparql_output = sparql_output.split('\n', 1)[1]
    if sparql_output.endswith('```'):
        sparql_output = sparql_output.rsplit('\n', 1)[0]
    sparql_output = sparql_output.strip()
    
    # Validate syntax
    try:
        validated = atlas_sparql.validate(sparql_output)
    except Exception as e:
        raise ValueError(f'LLM generated invalid SPARQL: {e}\n\nGenerated query:\n{sparql_output}')
    
    # SHACL pre-check: reject any INSERT/UPDATE queries
    if 'INSERT' in sparql_output.upper() or 'DELETE' in sparql_output.upper():
        raise ValueError(
            'SHACL pre-check FAILED: LLM generated a write query. '
            'The NL-to-SPARQL component is read-only. Write operations '
            'are prohibited at this layer.'
        )
    
    return validated

print('nl_to_sparql() function defined.')
print('Ready to translate natural-language questions to SPARQL.')

## Testing: Ask the Competency Questions in Plain English

The cell below sends each of the seven competency questions through the
NL-to-SPARQL component and validates the output. Every generated query must:
1. Parse as valid SPARQL (syntax check)
2. Use only atlas: vocabulary (grounding check)
3. Not contain INSERT/UPDATE (SHACL pre-check)
4. Return a non-empty result shape (semantic check against the ontology)

In [ ]:
# Test all seven competency questions
test_questions = [
    'Which customers have generated a wealth signal in the last 90 days?',
    'For signal S-001, what observations support it and what is the score decomposition?',
    'Which household relationships does customer C-001 have?',
    'Has customer C-001 been previously surfaced as a wealth candidate?',
    'Which routing decisions required human review and what was the outcome?',
    'What is the audit trail from signal detection to advisor approval for customer C-001?',
    'What specific transactions were used to surface customer C-001 as a candidate?',
]

print('NL-to-SPARQL Translation Test')
print('=' * 60)
print()

results = []
for i, question in enumerate(test_questions, 1):
    print(f'CQ{i}: {question}')
    try:
        sparql = nl_to_sparql(question)
        print(f'  [PASS] Valid SPARQL generated ({len(sparql)} chars)')
        # Show first 2 lines of the query body (after prefixes)
        body_lines = [l for l in sparql.split('\n') if not l.startswith('PREFIX') and l.strip()]
        for line in body_lines[:2]:
            print(f'         {line.strip()}')
        results.append(True)
    except ValueError as e:
        print(f'  [FAIL] {str(e)[:100]}')
        results.append(False)
    except Exception as e:
        print(f'  [ERROR] {type(e).__name__}: {str(e)[:100]}')
        results.append(False)
    print()

passed = sum(results)
total = len(results)
print(f'Results: {passed}/{total} questions translated successfully')

## Module 7 Validation Gate

The gate checks:
1. The NL-to-SPARQL function is defined and callable
2. At least 5 of 7 competency questions produce valid SPARQL
3. No generated query contains INSERT/UPDATE (SHACL pre-check)
4. The ground-truth pairs file exists with at least 7 entries
5. Bedrock is accessible and responds

In [ ]:
print('=' * 60)
print('MODULE 7 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Function defined
if callable(nl_to_sparql):
    print('[PASS] Gate 1 - nl_to_sparql() function is defined and callable')
else:
    print('[FAIL] Gate 1 - nl_to_sparql() not defined')
    gate_pass = False

# Gate 2: At least 5/7 questions produce valid SPARQL
if passed >= 5:
    print(f'[PASS] Gate 2 - {passed}/7 questions produce valid SPARQL (threshold: 5)')
else:
    print(f'[FAIL] Gate 2 - Only {passed}/7 questions produce valid SPARQL (threshold: 5)')
    gate_pass = False

# Gate 3: Pre-check actively rejects write attempts.
# This is an adversarial test: we deliberately pass known-bad queries to
# atlas_sparql.validate() and confirm each one is rejected. This verifies
# the SHACL pre-check is operational — not just claimed.
from atlas_sparql import AtlasSPARQLError as _AtlasSPARQLErr

adversarial_queries = [
    ('INSERT DATA', 'INSERT DATA { <https://github.com/your-org/atlas/instance#bad> <https://github.com/your-org/atlas/ontology#scoreValue> "0.99" }'),
    ('DELETE WHERE', 'DELETE WHERE { ?s ?p ?o }'),
    ('DROP GRAPH', 'DROP GRAPH <https://github.com/your-org/atlas/instance#g1>'),
    ('Probabilistic-opaque INSERT', 'PREFIX atlas: <https://github.com/your-org/atlas/ontology#> INSERT DATA { atlas:bad atlas:probabilisticOpaque true }'),
]

rejected = 0
for label, bad_query in adversarial_queries:
    try:
        from atlas_sparql import validate as _validate
        _validate(bad_query)
        print(f'[FAIL] Gate 3 - Adversarial query "{label}" was NOT rejected')
        gate_pass = False
    except _AtlasSPARQLErr:
        rejected += 1

if rejected == len(adversarial_queries):
    print(f'[PASS] Gate 3 - SHACL pre-check rejected {rejected}/{len(adversarial_queries)} adversarial write attempts')
else:
    print(f'[FAIL] Gate 3 - Pre-check rejected only {rejected}/{len(adversarial_queries)} adversarial queries')
    gate_pass = False

# Gate 4: Ground-truth pairs
if len(GROUND_TRUTH) >= 7:
    print(f'[PASS] Gate 4 - {len(GROUND_TRUTH)} ground-truth pairs defined (threshold: 7)')
else:
    print(f'[FAIL] Gate 4 - Only {len(GROUND_TRUTH)} ground-truth pairs')
    gate_pass = False

# Gate 5: Bedrock accessible
try:
    test_resp = bedrock.invoke_model(
        modelId=BEDROCK_MODEL,
        contentType='application/json',
        accept='application/json',
        body=json.dumps({
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': 10,
            'messages': [{'role': 'user', 'content': 'Say OK'}]
        })
    )
    print(f'[PASS] Gate 5 - Bedrock accessible ({BEDROCK_MODEL})')
except Exception as e:
    print(f'[FAIL] Gate 5 - Bedrock not accessible: {e}')
    gate_pass = False

print()
if gate_pass:
    print('MODULE 7 VALIDATION: PASS')
    print('You may proceed to Module 8.')
else:
    print('MODULE 7 VALIDATION: FAIL')
    raise AssertionError('Module 7 validation gate failed.')

## Extending This to Your Data

### When to add a competency question to ground-truth vs refine tips

- **Add to ground-truth** when the LLM consistently generates wrong property names
  or wrong query structure for a specific question pattern. A new example teaches
  the correct pattern.
- **Refine tips** when the LLM gets the structure right but uses wrong vocabulary
  across many questions. Tips are global hints; ground-truth is per-pattern.

### Handling hallucinated properties

The LLM may generate `atlas:hasWealth` (does not exist) instead of
`atlas:producesSignal` (correct). The SHACL pre-check does not catch this because
it is a read query, not a write. The fix:

1. The `atlas_sparql.validate()` function parses the query syntactically
2. Add a vocabulary check: extract all `atlas:` terms from the generated query
   and verify each exists in `atlas-core.ttl`
3. If a hallucinated term is found, return an error to the user: "The system
   generated a query using 'atlas:hasWealth' which does not exist in the ontology.
   Please rephrase your question."

### Swapping Bedrock models

If your region or governance constraints require a different model:
1. Change `BEDROCK_MODEL` to the new model ID
2. Test all ground-truth pairs against the new model
3. Adjust `max_tokens` if the new model generates longer/shorter responses
4. The prompt scaffold and validation logic do not change

## What Changed

| Artifact | Location | Description |
|----------|----------|-------------|
| NL-to-SPARQL function | This notebook | Translates plain English to validated SPARQL via Bedrock |
| Ground-truth pairs | This notebook (cell 4) | 7 competency-question/SPARQL pairs for few-shot prompting |
| SHACL pre-check | Built into nl_to_sparql() | Rejects any LLM-generated write queries |
| Prompt scaffold | Built into nl_to_sparql() | Ontology vocabulary, rules, and examples |

**Key architectural point established:**

The LLM is now in the architecture — but in a sharply circumscribed role. It
translates between human language and structured queries. Every output is validated
before execution. The LLM does not reason, score, or make decisions. This is the
posture that makes Bedrock defensible in a regulated architecture.

**What Module 8 builds on this:**

Module 8 stitches everything together into the end-to-end wealth-signal demo:
detect a signal, score it, route it through a bounded agent, surface it to a
human reviewer (Alex Morgan), and record the full audit trail in the SLGD. The
NL-to-SPARQL component from this module becomes the query interface for the
reviewer UI.